In [157]:
import pandas as pd
import numpy as np

In [158]:
pd.set_option("display.max_columns", None)

In [159]:
train = pd.read_csv('labels_day_train_features.csv', sep = ';').drop('date', axis=1)
test = pd.read_csv('labels_day_test_features.csv', sep = ';').drop('date', axis=1)

In [160]:
train = train.drop(columns=['date'], errors="ignore")
test = test.drop(columns=['date'], errors="ignore")

In [161]:
def wape(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    return f'{(1 - (abs(y_true - y_pred).sum() / abs(y_true).sum())):.4f}'

In [162]:
train.head()

,route,hour,boardings,Spartak,Spartak_peak,Locomotive,Locomotive_peak,Cska,Cska_peak,Dinamo,Dinamo_peak,Dinamo_h,Dinamo_h_peak,Cska_h,Cska_h_peak,Spartak_h,Spartak_h_peak,temperature_2m (°C),apparent_temperature (°C),precipitation (mm),rain (mm),pressure_msl (hPa),wind_speed_10m (km/h),wind_gusts_10m (km/h),snowfall_mm,snow_depth_mm,wind_direction_sin,wind_direction_cos,weather_clear,weather_cloudy,weather_fog,weather_drizzle,weather_rain,weather_snow,weather_thunder,day_off,public_holiday,working_weekend,short_day,school_holiday,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos,dayofweek,day
0,1,5,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-4.7,-9.4,0.0,0.0,1013.2,12.5,27.4,0.0,280.0,-0.992546,0.121869,0,1,0,0,0,0,0,1,1,0,0,1,0.965926,2.588190e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
1,1,6,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-5.4,-9.9,0.0,0.0,1013.4,10.8,25.9,0.0,280.0,-0.997564,-0.069756,0,1,0,0,0,0,0,1,1,0,0,1,1.000000,6.123234e-17,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
2,1,7,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.2,-10.8,0.0,0.0,1013.8,10.6,22.3,0.0,280.0,-0.951057,-0.309017,0,1,0,0,0,0,0,1,1,0,0,1,0.965926,-2.588190e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
3,1,8,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.8,-11.2,0.0,0.0,1013.5,8.9,21.2,0.0,280.0,-0.848048,-0.529919,0,1,0,0,0,0,0,1,1,0,0,1,0.866025,-5.000000e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
4,1,9,53,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.5,-11.0,0.0,0.0,1013.2,9.6,19.8,0.0,280.0,-0.788011,-0.615661,0,1,0,0,0,0,0,1,1,0,0,1,0.707107,-7.071068e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1


In [163]:
nums_col = ["temperature_2m (°C)", "apparent_temperature (°C)",
            "precipitation (mm)", "rain (mm)", "snowfall_mm",
            "snow_depth_mm", "pressure_msl (hPa)",
            "wind_speed_10m (km/h)", "wind_gusts_10m (km/h)",
            ]


from pathlib import Path
import joblib
from sklearn.preprocessing import MinMaxScaler

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
df_all = pd.concat([train, test], ignore_index=True)

scaler = MinMaxScaler()
# df_all[nums_col] = scaler.fit_transform(df_all[nums_col])
train[nums_col] = scaler.fit_transform(train[nums_col])
test[nums_col] = scaler.transform(test[nums_col])
joblib.dump(scaler, ARTIFACTS_DIR / "minmax_scaler.joblib")


['artifacts\\minmax_scaler.joblib']

In [164]:
train_routes = sorted(df_all['route'].dropna().unique())
# train_routes = sorted(train['route'].dropna().unique())

test_datasets_by_route = {
    route: test[test['route'] == route].copy()
    for route in train_routes
}

datasets_by_route = {
    route: train[train['route'] == route].copy()
    for route in train_routes
}

# datasets_by_route = {
#     route: df_all[df_all['route'] == route].copy()
#     for route in train_routes
# }

# for route in train_routes:
#     print(f"?????? {route}: train={len(datasets_by_route[route])}, test={len(test_datasets_by_route[route])}")


In [ ]:
import matplotlib.pyplot as plt
import joblib
import catboost as ctb

nums_all = ["temperature_2m (°C)", "apparent_temperature (°C)",
            "precipitation (mm)", "rain (mm)", "snowfall_mm",
            "snow_depth_mm", "pressure_msl (hPa)",
            "wind_speed_10m (km/h)", "wind_gusts_10m (km/h)",
            'wind_direction_sin', 'wind_direction_cos',
            'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos',
            'month_sin', 'month_cos', 'dayofyear_sin',	'dayofyear_cos',
            "boardings", "route"
            ]

cat_cols = [i for i in train.columns.to_list() if i not in nums_all]
models = {}
midlle = 0

for route in train_routes:
    Y = datasets_by_route[route]["boardings"]
    X = datasets_by_route[route].drop(columns=["boardings", "route"])
    Y_test = test_datasets_by_route[route]["boardings"]
    X_test = test_datasets_by_route[route].drop(columns=["boardings", "route"])

    model = ctb.CatBoostRegressor(
        loss_function="MAE",
        cat_features = cat_cols,
        eval_metric="MAE",
        iterations=500,
        learning_rate=0.05,
        depth=5,
        l2_leaf_reg=2,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        min_data_in_leaf = 100
        # loss_function="MAE",
        # cat_features = cat_cols,
        # eval_metric="MAE",
        # iterations=500,
        # learning_rate=0.05,
        # depth=8,
        # l2_leaf_reg=5,
        # random_seed=42,
        # verbose=False,
        # allow_writing_files=False,
    )

    model.fit(
        X, Y,
        verbose=False,
    )
   

    # models[route] = model
    # model_path = ARTIFACTS_DIR / f"catboost_route_{route}.joblib"
    # joblib.dump(model, model_path)

    y_pred = model.predict(X_test)
    midlle += float(wape(Y_test, y_pred))
    print(f"route {route}: {wape(Y_test, y_pred)};")

print(f'wape: {midlle / 9}')

route 1: 0.8531;
route 7: 0.7662;
route 11: 0.8796;
route 12: 0.9038;


In [ ]:
# # SARIMA: отдельная модель на каждый маршрут по почасовому ряду посадок.
# # Нужен пакет statsmodels: uv add statsmodels (в папке mp).
# import warnings

# from statsmodels.tsa.statespace.sarimax import SARIMAX

# SARIMA_ORDER = (1, 0, 1)
# SARIMA_SEASONAL_ORDER = (1, 1, 1, 24)  # суточная сезонность: 24 часа

# # внешние признаки (тогда это SARIMAX); пустой список - чистая SARIMA.
# # Суточная сезонность не отличает выходные от будних, поэтому по умолчанию передаётся day_off
# EXOG_COLUMNS = ["day_off"]

# # train и test уже прошли через MinMaxScaler (ячейка с nums_col), но без date.
# # Даты для временного ряда берутся из тех же CSV: порядок строк тот же
# sarima_train = train.copy()
# sarima_test = test.copy()
# sarima_train["date"] = pd.read_csv("labels_day_train_features.csv", sep=";", usecols=["date"])["date"].to_numpy()
# sarima_test["date"] = pd.read_csv("labels_day_test_features.csv", sep=";", usecols=["date"])["date"].to_numpy()


# def hourly_index(data):
#     return pd.to_datetime(data["date"]) + pd.to_timedelta(data["hour"], unit="h")


# def full_hourly_index(data):
#     dates = pd.to_datetime(data["date"])
#     return pd.date_range(dates.min(), dates.max() + pd.Timedelta(hours=23), freq="h")


# def exog_by_hour(data):
#     """Внешние признаки на полной почасовой сетке.

#     Календарь, погода и матчи одинаковы для всех маршрутов, поэтому берутся из всего набора:
#     так они есть и для часов (и даже дней), когда у конкретного маршрута не было посадок.
#     Часы, где посадок не было ни на одном маршруте, заполняются соседними часами того же дня.
#     """
#     index = full_hourly_index(data)
#     if not EXOG_COLUMNS:
#         return pd.DataFrame(index=index)
#     exog = data.set_index(hourly_index(data))[EXOG_COLUMNS]
#     exog = exog[~exog.index.duplicated()].reindex(index)
#     return exog.groupby(exog.index.normalize()).transform(lambda column: column.ffill().bfill())


# def to_hourly_frame(route_data, exog):
#     """Полная почасовая сетка маршрута: часов без посадок в разметке нет, для них boardings = 0."""
#     boardings = route_data.set_index(hourly_index(route_data))["boardings"]
#     frame = exog.copy()
#     frame["boardings"] = boardings.reindex(exog.index).fillna(0)
#     return frame


# train_exog = exog_by_hour(sarima_train)
# test_exog = exog_by_hour(sarima_test)

# sarima_models = {}
# sarima_predictions = {}

# for route in sorted(sarima_train["route"].unique()):
#     route_train = sarima_train[sarima_train["route"] == route]
#     route_test = sarima_test[sarima_test["route"] == route]

#     history = to_hourly_frame(route_train, train_exog)
#     future = to_hourly_frame(route_test, test_exog)
#     if future.index[0] != history.index[-1] + pd.Timedelta(hours=1):
#         raise ValueError(f"Маршрут {route}: test не продолжает train без разрыва")

#     model = SARIMAX(
#         history["boardings"],
#         exog=history[EXOG_COLUMNS] if EXOG_COLUMNS else None,
#         order=SARIMA_ORDER,
#         seasonal_order=SARIMA_SEASONAL_ORDER,
#         enforce_stationarity=False,
#         enforce_invertibility=False,
#     )
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         result = model.fit(disp=False)

#     forecast = result.forecast(
#         steps=len(future),
#         exog=future[EXOG_COLUMNS] if EXOG_COLUMNS else None,
#     )
#     # посадки не бывают отрицательными
#     forecast = pd.Series(np.clip(np.asarray(forecast), 0, None), index=future.index)

#     # оценка только по строкам, которые есть в разметке test (как у LightGBM)
#     y_pred = forecast.reindex(hourly_index(route_test)).to_numpy()
#     sarima_models[route] = result
#     sarima_predictions[route] = y_pred
#     print(f"route {route}: {wape(route_test['boardings'], y_pred)}; AIC {result.aic:.0f}")

In [ ]:
train.head()

,route,hour,boardings,Spartak,Spartak_peak,Locomotive,Locomotive_peak,Cska,Cska_peak,Dinamo,Dinamo_peak,Dinamo_h,Dinamo_h_peak,Cska_h,Cska_h_peak,Spartak_h,Spartak_h_peak,temperature_2m (°C),apparent_temperature (°C),precipitation (mm),rain (mm),pressure_msl (hPa),wind_speed_10m (km/h),wind_gusts_10m (km/h),snowfall_mm,snow_depth_mm,wind_direction_sin,wind_direction_cos,weather_clear,weather_cloudy,weather_fog,weather_drizzle,weather_rain,weather_snow,weather_thunder,day_off,public_holiday,working_weekend,short_day,school_holiday,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,month_sin,month_cos,dayofyear_sin,dayofyear_cos,dayofweek,day
0,1,5,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-4.7,-9.4,0.0,0.0,1013.2,12.5,27.4,0.0,280.0,-0.992546,0.121869,0,1,0,0,0,0,0,1,1,0,0,1,0.965926,2.588190e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
1,1,6,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-5.4,-9.9,0.0,0.0,1013.4,10.8,25.9,0.0,280.0,-0.997564,-0.069756,0,1,0,0,0,0,0,1,1,0,0,1,1.000000,6.123234e-17,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
2,1,7,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.2,-10.8,0.0,0.0,1013.8,10.6,22.3,0.0,280.0,-0.951057,-0.309017,0,1,0,0,0,0,0,1,1,0,0,1,0.965926,-2.588190e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
3,1,8,26,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.8,-11.2,0.0,0.0,1013.5,8.9,21.2,0.0,280.0,-0.848048,-0.529919,0,1,0,0,0,0,0,1,1,0,0,1,0.866025,-5.000000e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1
4,1,9,53,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-6.5,-11.0,0.0,0.0,1013.2,9.6,19.8,0.0,280.0,-0.788011,-0.615661,0,1,0,0,0,0,0,1,1,0,0,1,0.707107,-7.071068e-01,0.974928,-0.222521,0.0,1.0,0.0,1.0,2,1


In [ ]:
# import matplotlib.pyplot as plt

# def plot_boxplots(dataframe, columns=None, ncols=3):
#     """Построить boxplot для каждого числового столбца DataFrame."""
#     if not isinstance(dataframe, pd.DataFrame):
#         raise TypeError("dataframe должен быть pandas.DataFrame")

#     if columns is None:
#         columns = dataframe.select_dtypes(include=np.number).columns.tolist()
#     else:
#         columns = list(columns)

#     if not columns:
#         raise ValueError("В DataFrame нет числовых столбцов для визуализации")

#     missing_columns = set(columns) - set(dataframe.columns)
#     if missing_columns:
#         raise ValueError(f"Столбцы отсутствуют в DataFrame: {sorted(missing_columns)}")

#     ncols = min(ncols, len(columns))
#     nrows = (len(columns) + ncols - 1) // ncols
#     fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
#     axes = np.atleast_1d(axes).ravel()

#     for axis, column in zip(axes, columns):
#         axis.boxplot(dataframe[column].dropna(), vert=True)
#         axis.set_title(str(column))
#         axis.set_ylabel("Значение")
#         axis.grid(axis="y", alpha=0.25)

#     for axis in axes[len(columns):]:
#         axis.remove()

#     fig.tight_layout()
#     return fig, axes


# fig, axes = plot_boxplots(train)
